In [ ]:
from google.colab import drive
drive.mount("/content/drive")
D = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT"
!ls "{D}"
!ls "{D}/xbd_tars" "{D}/xbd"

Mounted at /content/drive
 checkpoints  'SEM 1 FINAL PROJECT'   xbd   xbd_tars
'/content/drive/MyDrive/MAAI 902 X 901 PROJECT/xbd':
flood_splits.csv      hold	     rscc_flood_manifest.csv  tier3
flood_tile_stats.csv  processed.zip  test		      train

'/content/drive/MyDrive/MAAI 902 X 901 PROJECT/xbd_tars':
test_images_labels_targets.tar	train_images_labels_targets.tar


In [ ]:
!git clone -q https://github.com/aaarrvind/FloodVerify.git /content/FloodVerify
%cd /content/FloodVerify
!pip install -q -e . --no-deps jsonschema
import sys; sys.path.insert(0, "/content/FloodVerify/src")

/content/FloodVerify
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for floodsitrep (pyproject.toml) ... done


In [ ]:
!mkdir -p /content/xbd/processed
!tar -xf "{D}/xbd_tars/train_images_labels_targets.tar" -C /content/xbd
!tar -xf "{D}/xbd_tars/test_images_labels_targets.tar" -C /content/xbd
!cp -r "{D}/xbd/hold" /content/xbd/
!cp -r "{D}/xbd/tier3" /content/xbd/
!cp "{D}/xbd/rscc_flood_manifest.csv" /content/xbd/
!unzip -q "{D}/xbd/processed.zip" -d /content/xbd/processed
!ls /content/xbd
!ls /content/xbd/processed/evidence_labels | wc -l

hold  processed  rscc_flood_manifest.csv  test	tier3  train
10148


In [ ]:
CK = f"{D}/checkpoints"
!mkdir -p "{CK}"


In [ ]:
from floodsitrep.data.dataset import XbdTiles
from torch.utils.data import DataLoader
X, MAN = "/content/xbd", "/content/xbd/rscc_flood_manifest.csv"
for role in ("train", "val", "test"):
    print(role, len(XbdTiles(X, MAN, role=role)))
b = next(iter(DataLoader(XbdTiles(X, MAN, role="train", augment=True), batch_size=8, shuffle=True)))
print(b["pre"].shape, b["mask"].shape, b["mask"].unique().tolist())
print("share of valid pixels:", b["valid"].float().mean().item())
print("pixels per class:", [(b["mask"] == k).sum().item() for k in range(5)])

train 5488
val 2184
test 2476
torch.Size([8, 3, 512, 512]) torch.Size([8, 512, 512]) [0, 1, 2, 3]
share of valid pixels: 1.0
pixels per class: [1889294, 179463, 23157, 5238, 0]


In [ ]:
%cd /content/FloodVerify
!git fetch -q origin && git checkout -q m2/models-train && git log --oneline -3

/content/FloodVerify
72b9e61 (HEAD -> m2/models-train, origin/m2/models-train) M0/M1/M2 models, Dice+focal/CE losses, resumable AMP train.py, run configs, Colab notebook
a18ce27 (origin/m3/eval-metrics) cv/eval/metrics: pixel-level per-class IoU/F1, xView2-style score, tile bootstrap
28f642a Merge pull request #9 from aaarrvind/m1/readme-colab-note


In [ ]:
import torch
from floodsitrep.cv.models import build_model
m = build_model({"model": "siamese_unet"}).cuda().eval()
with torch.no_grad(), torch.autocast("cuda"):
    out = m(b["pre"][:2].cuda(), b["post"][:2].cuda())
print(out.shape, sum(p.numel() for p in m.parameters()) / 1e6, "M parameters")

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 238MB/s]


torch.Size([2, 5, 512, 512]) 25.495765 M parameters


In [ ]:
import torch, torch.nn.functional as F
from floodsitrep.cv.models.losses import SegLoss
with torch.no_grad(), torch.autocast("cuda"):
    logits = m(b["pre"].cuda(), b["post"].cuda()).float()
tgt = b["mask"].cuda().clone()
tgt[~b["valid"].cuda()] = 255                     # no-data -> ignored
pix = F.cross_entropy(logits, tgt, ignore_index=255, reduction="none")
total = pix[tgt != 255].sum()
for k, name in enumerate(["background", "no damage", "minor", "major", "destroyed"]):
    sel = tgt == k
    print(f"{name:11s} pixels {int(sel.sum()):8d}   share of plain CE loss {(pix[sel].sum() / total).item():.4f}")
w = [0.10, 0.52, 0.88, 0.75, 2.74]
for kind in ("ce", "focal", "dice", "dice_focal"):
    print(kind, round(SegLoss(kind, class_weights=w)(logits, tgt).item(), 4))

background  pixels  1889294   share of plain CE loss 0.9058
no damage   pixels   179463   share of plain CE loss 0.0805
minor       pixels    23157   share of plain CE loss 0.0112
major       pixels     5238   share of plain CE loss 0.0024
destroyed   pixels        0   share of plain CE loss 0.0000
ce 1.6115
focal 0.151
dice 0.9067
dice_focal 1.0577


In [ ]:
%cd /content/FloodVerify
!git checkout -q m2/models-train && git branch --show-current

/content/FloodVerify
m2/models-train


In [ ]:
D = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT"
CK = f"{D}/checkpoints"
X, MAN = "/content/xbd", "/content/xbd/rscc_flood_manifest.csv"
!python -m floodsitrep.cv.train --config configs/cv/smoke_m1.yaml --xbd-root {X} --manifest {MAN} --out-dir "{CK}"
!ls -la "{CK}/smoke_m1_s0"

run smoke_m1_s0: train 320 tiles, val 96 tiles, device cuda, amp True
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth
100% 83.3M/83.3M [00:00<00:00, 241MB/s]
resumed from /content/drive/MyDrive/MAAI 902 X 901 PROJECT/checkpoints/smoke_m1_s0/last.pt at epoch 2 (best so far 0.1566)
total 398886
drwx------ 2 root root      4096 Nov 15 19:29 .
drwx------ 3 root root      4096 Nov 15 13:42 ..
-rw------- 1 root root 102175063 Nov 15 15:51 best.pt
-rw------- 1 root root       419 Nov 15 19:29 config.yaml
-rw------- 1 root root 306272821 Nov 15 15:52 last.pt
-rw------- 1 root root      1736 Nov 15 15:51 metrics.jsonl


In [ ]:
!python -m floodsitrep.cv.train --config configs/cv/m1_siamese_unet.yaml --xbd-root {X} --manifest {MAN} --out-dir "{CK}" --set num_workers=8

run m1_siamese_unet_s0: train 5488 tiles, val 2184 tiles, device cuda, amp True
  epoch 0 step 50/686 loss 0.9847 (running 1.0076)
  epoch 0 step 100/686 loss 0.8917 (running 0.9777)
  epoch 0 step 150/686 loss 0.8466 (running 0.9451)
  epoch 0 step 200/686 loss 0.8481 (running 0.9171)
  epoch 0 step 250/686 loss 0.7725 (running 0.8938)
  epoch 0 step 300/686 loss 0.8998 (running 0.8750)
  epoch 0 step 350/686 loss 0.7197 (running 0.8613)
  epoch 0 step 400/686 loss 0.7194 (running 0.8478)
  epoch 0 step 450/686 loss 0.7511 (running 0.8364)
  epoch 0 step 500/686 loss 0.7429 (running 0.8261)
  epoch 0 step 550/686 loss 0.7341 (running 0.8171)
  epoch 0 step 600/686 loss 0.6473 (running 0.8093)
  epoch 0 step 650/686 loss 0.7660 (running 0.8012)
epoch 0: train_loss 0.7969 val_loss 0.7285 xview2_pixel 0.2395 loc_f1 0.6237 dmg_f1 minor 0.033 major 0.340 destroyed 0.055 (216.2s)
  epoch 1 step 50/686 loss 0.6747 (running 0.6954)
  epoch 1 step 100/686 loss 0.6110 (running 0.7030)
  epoch 1

In [ ]:
from google.colab import runtime
runtime.unassign()

In [ ]:
!python -m floodsitrep.cv.train --config configs/cv/m1_siamese_unet_ce.yaml --xbd-root {X} --manifest {MAN} --out-dir "{CK}" --set num_workers=8

run m1_siamese_unet_ce_s0: train 5488 tiles, val 2184 tiles, device cuda, amp True
  epoch 0 step 50/686 loss 1.3517 (running 1.4182)
  epoch 0 step 100/686 loss 0.8745 (running 1.3001)
  epoch 0 step 150/686 loss 0.4343 (running 1.1599)
  epoch 0 step 200/686 loss 0.8342 (running 1.0541)
  epoch 0 step 250/686 loss 0.7718 (running 0.9717)
  epoch 0 step 300/686 loss 1.1922 (running 0.9092)
  epoch 0 step 350/686 loss 0.3172 (running 0.8587)
  epoch 0 step 400/686 loss 0.4507 (running 0.8216)
  epoch 0 step 450/686 loss 0.4691 (running 0.7863)
  epoch 0 step 500/686 loss 0.3962 (running 0.7613)
  epoch 0 step 550/686 loss 0.4764 (running 0.7418)
  epoch 0 step 600/686 loss 0.4137 (running 0.7242)
  epoch 0 step 650/686 loss 0.4795 (running 0.7084)
epoch 0: train_loss 0.6972 val_loss 0.2181 xview2_pixel 0.1751 loc_f1 0.5784 dmg_f1 minor 0.001 major 0.237 destroyed 0.073 (204.6s)
  epoch 1 step 50/686 loss 0.1743 (running 0.5027)
  epoch 1 step 100/686 loss 0.3274 (running 0.5082)
  epoc

In [ ]:
!python -m floodsitrep.cv.train --config configs/cv/m0_early_fusion_unet.yaml --xbd-root {X} --manifest {MAN} --out-dir "{CK}" --set num_workers=8

run m0_early_fusion_unet_s0: train 5488 tiles, val 2184 tiles, device cuda, amp True
  epoch 0 step 50/686 loss 0.9893 (running 1.0460)
  epoch 0 step 100/686 loss 0.9245 (running 1.0063)
  epoch 0 step 150/686 loss 0.8775 (running 0.9709)
  epoch 0 step 200/686 loss 0.8796 (running 0.9405)
  epoch 0 step 250/686 loss 0.7895 (running 0.9174)
  epoch 0 step 300/686 loss 0.8537 (running 0.8982)
  epoch 0 step 350/686 loss 0.7271 (running 0.8843)
  epoch 0 step 400/686 loss 0.7302 (running 0.8715)
  epoch 0 step 450/686 loss 0.7336 (running 0.8602)
  epoch 0 step 500/686 loss 0.7408 (running 0.8510)
  epoch 0 step 550/686 loss 0.7029 (running 0.8428)
  epoch 0 step 600/686 loss 0.7793 (running 0.8371)
  epoch 0 step 650/686 loss 0.7440 (running 0.8305)
epoch 0: train_loss 0.8268 val_loss 0.7304 xview2_pixel 0.1928 loc_f1 0.6425 dmg_f1 minor 0.033 major 0.166 destroyed 0.000 (157.8s)
  epoch 1 step 50/686 loss 0.7108 (running 0.7607)
  epoch 1 step 100/686 loss 0.7229 (running 0.7605)
  ep

In [ ]:
from google.colab import runtime
runtime.unassign()